# **bfloat16 Support in kmeans_tpu**

This notebook demonstrates the bfloat16 data type support in `kmeans_tpu`.
bfloat16 (Brain Float 16) is a 16-bit format developed by Google specifically
for TPU workloads. Compared to float32, it uses half the memory whilst
preserving the same exponent range, making it a practical option for
large datasets where TPU memory is a constraint.

> **Note:** This notebook requires a TPU runtime.
> In Colab: Runtime → Change runtime type → TPU v5e-1 TPU

> **Compatibility:** This notebook has been tested with JAX 0.7.2 and
> Python 3.13 on Google Colab.

## 1. Installation and setup

In [ ]:
!pip install git+https://github.com/vbrachetta/kmeans_tpu.git

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from kmeans_tpu import kmeans, inertia

print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")

## 2. Generate data

We use `make_blobs` to generate a simple 2D dataset with 3 well-separated
clusters.

In [ ]:
data_np, true_labels = make_blobs(
    n_samples=300,
    n_features=2,
    centers=3,
    random_state=42,
)

data_f32  = jnp.array(data_np.astype("float32"))
data_bf16 = data_f32.astype(jnp.bfloat16)

print(f"Dataset  : {data_np.shape[0]} samples, {data_np.shape[1]} features")
print(f"float32  : dtype={data_f32.dtype},  memory={data_f32.nbytes} bytes")
print(f"bfloat16 : dtype={data_bf16.dtype}, memory={data_bf16.nbytes} bytes")
print(f"Memory reduction: {(1 - data_bf16.nbytes / data_f32.nbytes) * 100:.0f}%")

## 3. Run clustering with both dtypes

In [ ]:
centroids_f32,  assignments_f32,  n_iter_f32  = kmeans(data_f32,  k=3, seed=42)
centroids_bf16, assignments_bf16, n_iter_bf16 = kmeans(data_bf16, k=3, seed=42)

score_f32  = inertia(data_f32, centroids_f32, assignments_f32)
score_bf16 = inertia(data_f32, centroids_f32.astype(jnp.float32), assignments_bf16)

diff_pct = abs(score_f32 - score_bf16) / score_f32 * 100

print(f"float32  — Inertia: {score_f32:.2f},  Iterations: {n_iter_f32}")
print(f"bfloat16 — Inertia: {score_bf16:.2f}, Iterations: {n_iter_bf16}")
print(f"Inertia difference: {diff_pct:.4f}%")

## 4. Visual comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(data_np[:, 0], data_np[:, 1],
                c=np.array(assignments_f32), s=20, alpha=0.7)
axes[0].scatter(np.array(centroids_f32)[:, 0],
                np.array(centroids_f32)[:, 1],
                c="black", s=200, marker="X", zorder=5, label="Centroids")
axes[0].set_title(f"float32 (inertia={score_f32:.0f}, iter={n_iter_f32})")
axes[0].legend()

axes[1].scatter(data_np[:, 0], data_np[:, 1],
                c=np.array(assignments_bf16), s=20, alpha=0.7)
axes[1].scatter(np.array(centroids_bf16.astype(jnp.float32))[:, 0],
                np.array(centroids_bf16.astype(jnp.float32))[:, 1],
                c="black", s=200, marker="X", zorder=5, label="Centroids")
axes[1].set_title(f"bfloat16 (inertia={score_bf16:.0f}, iter={n_iter_bf16})")
axes[1].legend()

plt.suptitle("float32 vs bfloat16 — Clustering Results", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Summary

bfloat16 uses half the memory of float32 with negligible impact on clustering quality on this dataset, and may be worth considering for large datasets where TPU memory is a constraint.